## Hyperparameters setting

In [1]:
import os
import pandas as pd
import ast

df_param = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning", "results.csv"))

## Model run preparation

Import data

In [2]:
import os
import pandas as pd

df = pd.read_csv(os.path.join(os.path.dirname(os.getcwd()), "data", "data_advice_fulltext.csv"))
docs = list(df["text"])
docs = [doc.replace('\xa0', '') for doc in docs]
classes = list(df["gen"])
id = list(df["ID"])

Pre-load embeddings

Prepare topic fine-tuning models

In [3]:
from bertopic.representation import KeyBERTInspired
from bertopic.representation import MaximalMarginalRelevance

# The main representation of a topic
main_representation = KeyBERTInspired()

# Additional ways of representing a topic
aspect_model2 = [KeyBERTInspired(top_n_words=20), MaximalMarginalRelevance(diversity=.5)]

# Add all models together to be run in a single `fit`
representation_model = {
   "KeyBERT": main_representation,
   "MMR":  aspect_model2 
}

## Model run

Train model

In [4]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from evaluation import evaluate_model
import ast

for i, (param, run_name)  in enumerate(zip(df_param["params"], df_param["run_name"])):
    # Pre-calculate embeddings
    embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", use_auth_token=False)
    embeddings = embedding_model.encode(docs, show_progress_bar=True)
    
    param_dic = ast.literal_eval(param)
    
    if "max_df" not in param_dic:
        param_dic["max_df"] = 1.0
    
    # Data saving

    data = []


    # Setup different models

    cluster_model = HDBSCAN(min_cluster_size=param_dic["min_cluster_size"], metric='euclidean', cluster_selection_method='eom', prediction_data=True)
    vectorizer_model = CountVectorizer(stop_words="english", min_df=param_dic["min_df"], max_df=param_dic["max_df"], ngram_range=param_dic["ngram_range"])
    ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)

    for seed in param_dic["random_state"]:

        umap_model = UMAP(n_neighbors=param_dic["n_neighbors"], n_components=param_dic["n_components"], min_dist=param_dic["min_dist"], metric='cosine', random_state=seed)

        topic_model = BERTopic(

            # Pipeline models
            embedding_model=embedding_model,
            umap_model=umap_model,
            hdbscan_model=cluster_model,
            vectorizer_model=vectorizer_model,
            representation_model=representation_model,

            # Hyperparameters
            top_n_words=param_dic["top_n_words"],
            n_gram_range=param_dic["ngram_range"],
            min_topic_size="auto", #use HDBSCAN
            verbose=True,

            # General parameters
            calculate_probabilities=True,
            language="english"
        )

        topics, probs = topic_model.fit_transform(docs, embeddings)

        eval_res = evaluate_model(topic_model, docs, topics, embeddings, topk=param_dic["top_n_words"])
        data.append([seed, max(topic_model.topics_)] + [i for i in eval_res])

    data = pd.DataFrame(data, columns=["seed", "nr_topic", "c_v", "c_npmi", "t_D", "silhouette", "similarity"])

    embedding_model = "all-MiniLM-L6-v2"
    topic_model.save(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning_correct", run_name), serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

    data_run = [[run_name, param_dic, data["nr_topic"].mean(), data["c_v"].mean(), data["c_npmi"].mean(), data["t_D"].mean(), data["silhouette"].mean(), data["similarity"].mean()]]

    df_run = pd.DataFrame(data_run, columns=["run_name", "params", "nr_topic", "c_v", "c_npmi", "diversity", "silhouette", "similarity"])

    df_run.to_csv(os.path.join(os.path.dirname(os.getcwd()), "results", "BERT_fine_tuning_correct", "results.csv"), mode="a", header=False, index=False)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2025-02-20 12:17:19,771 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 12:17:26,771 - BERTopic - Dimensionality - Completed ✓
2025-02-20 12:17:26,771 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 12:17:26,830 - BERTopic - Cluster - Completed ✓
2025-02-20 12:17:26,830 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 12:17:33,908 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:00<00:00,  3.00it/s]
2025-02-20 12:18:06,689 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 12:18:08,751 - BERTopic - Dimensionality - Completed ✓
2025-02-20 12:18:08,751 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 12:18:08,807 - BERTopic - Cluster - Completed ✓
2025-02-20 12:18:08,807 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 12:18:14,967 - BERTop

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2025-02-20 12:21:11,201 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 12:21:13,649 - BERTopic - Dimensionality - Completed ✓
2025-02-20 12:21:13,649 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 12:21:13,721 - BERTopic - Cluster - Completed ✓
2025-02-20 12:21:13,721 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 12:21:22,548 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:00<00:00,  2.60it/s]
2025-02-20 12:21:59,075 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 12:22:01,324 - BERTopic - Dimensionality - Completed ✓
2025-02-20 12:22:01,324 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 12:22:01,389 - BERTopic - Cluster - Completed ✓
2025-02-20 12:22:01,393 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 12:22:07,703 - BERTop

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

...

2025-02-20 16:37:42,882 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 16:37:45,291 - BERTopic - Dimensionality - Completed ✓
2025-02-20 16:37:45,291 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 16:37:45,339 - BERTopic - Cluster - Completed ✓
2025-02-20 16:37:45,342 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 16:37:49,805 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:00<00:00,  2.24it/s]
2025-02-20 16:38:18,835 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 16:38:21,167 - BERTopic - Dimensionality - Completed ✓
2025-02-20 16:38:21,167 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 16:38:21,222 - BERTopic - Cluster - Completed ✓
2025-02-20 16:38:21,222 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 16:38:26,136 - BERTop

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2025-02-20 16:40:58,917 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 16:41:01,332 - BERTopic - Dimensionality - Completed ✓
2025-02-20 16:41:01,332 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 16:41:01,370 - BERTopic - Cluster - Completed ✓
2025-02-20 16:41:01,384 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 16:41:05,634 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:00<00:00,  3.01it/s]
2025-02-20 16:41:35,294 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 16:41:37,643 - BERTopic - Dimensionality - Completed ✓
2025-02-20 16:41:37,643 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 16:41:37,696 - BERTopic - Cluster - Completed ✓
2025-02-20 16:41:37,696 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 16:41:42,140 - BERTop

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2025-02-20 16:44:17,892 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 16:44:20,222 - BERTopic - Dimensionality - Completed ✓
2025-02-20 16:44:20,222 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 16:44:20,282 - BERTopic - Cluster - Completed ✓
2025-02-20 16:44:20,284 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 16:44:24,573 - BERTopic - Representation - Completed ✓
100%|██████████| 1/1 [00:00<00:00,  3.15it/s]
2025-02-20 16:44:54,514 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2025-02-20 16:44:56,912 - BERTopic - Dimensionality - Completed ✓
2025-02-20 16:44:56,914 - BERTopic - Cluster - Start clustering the reduced embeddings
2025-02-20 16:44:56,958 - BERTopic - Cluster - Completed ✓
2025-02-20 16:44:56,961 - BERTopic - Representation - Extracting topics from clusters using representation models.
2025-02-20 16:45:01,729 - BERTop